# Etapa 4: Limpieza, Saneamiento Léxico y Gobernanza de Datos

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 4: Data Cleaning) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, DAMA-BOK, PEP 8, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Saneamiento Léxico**: Remoción de acentos, caracteres especiales y normalización en mayúsculas sostenidas.
2. **Estandarización de Variedades**: Mapeo de denominaciones comerciales dispersas a variedades canónicas (Pastusa, Capiro, Criolla, etc.).
3. **Desambiguación Territorial**: Resolución de municipios homónimos (ej. *La Unión* en Antioquia, Nariño y Valle).
4. **Persistencia Medallion**: Exportación a `data/CLEANED/dataset_sipsa_mensual_limpio.parquet`.

In [ ]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    fastparquet>=2023.8.0 \
    tabulate>=0.9.0

print(f"[OK] Ambiente verificado en Python: {sys.version.split()[0]}")

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DE IMPORTS Y RUTAS DEL PROYECTO (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.infrastructure.config import CLEANED_DIR
from src.application.cleaning_service import CleaningService

print("Servicio de Limpieza importado exitosamente.")

In [ ]:
# ==============================================================================
# CARGA DE DATOS VALIDADOS
# ==============================================================================
parquet_path = CLEANED_DIR / "dataset_sipsa_mensual_nacional.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    # Fallback con variabilidad léxica
    df = pd.DataFrame({
        "fecha_mes": ["2023-01-01", "2023-01-01", "2023-01-01", "2023-01-01"],
        "año": [2023, 2023, 2023, 2023],
        "mes": [1, 1, 1, 1],
        "divipola_depto": ["05", "52", "25", "15"],
        "nombre_depto": ["ANTIOQUIA", "NARIÑO", "CUNDINAMARCA", "BOYACÁ"],
        "divipola_mpio": ["05400", "52399", "25875", "15861"],
        "nombre_mpio": ["LA UNION", "LA UNION", "VILLAPINZÓN", "VENTAQUEMADA"],
        "mercado_mayorista": ["MEDELLIN, CMA", "PASTO, EL POTRERILLO", "CORABASTOS", "CORABASTOS"],
        "alimento": ["PAPA", "PAPA", "PAPA", "PAPA"],
        "variedad_papa": ["PAPA DIACOL CAPIRO", "PAPA R-12 NEGRA", "PAPA SUPREMA", "PAPA CRIOLLA LIMPIA"],
        "volumen_ingreso_ton": [250.0, 180.0, 310.0, 95.0],
        "precio_prom_kg": [2300.0, 1950.0, 2400.0, 3100.0],
        "precio_min_kg": [2200.0, 1800.0, 2300.0, 2900.0],
        "precio_max_kg": [2450.0, 2100.0, 2550.0, 3300.0],
        "num_transacciones": [40, 35, 50, 20]
    })

print(f"Registros listos para saneamiento: {len(df):,}")
display(df.head())

In [ ]:
# ==============================================================================
# EJECUCIÓN DEL SERVICIO DE SANEAMIENTO Y DESAMBIGUACIÓN
# ==============================================================================
clean_service = CleaningService(df, output_dir=CLEANED_DIR)
df_limpio = clean_service.ejecutar_limpieza()

print("=== DISTRIBUCIÓN DE VARIEDADES CANÓNICAS HOMOLOGADAS ===")
print(df_limpio["variedad_papa"].value_counts())

print("\n=== VERIFICACIÓN DE DESAMBIGUACIÓN DE HOMÓNIMOS (LA UNIÓN) ===")
display(df_limpio[df_limpio["nombre_mpio"].str.contains("LA UNION")][["divipola_depto", "nombre_depto", "divipola_mpio", "nombre_mpio"]])

### Conclusiones de la Etapa 4:
- Los textos han sido purgados de caracteres especiales y acentos.
- Las variedades de papa han quedado estandarizadas para evitar fragmentación en el modelado estadístico.
- El dataset saneado se encuentra persistido en `data/CLEANED/dataset_sipsa_mensual_limpio.parquet`, habilitando la **Etapa 5 (Features)**.